# 使用 SageMaker 进行超参优化
在超参优化的过程中，可能会存在多个局部高低点，不同的超参数也可能存在不同的寻优路径。SageMaker 本身提供了贝叶斯超参优化的功能，这对于一些超参数量多、模型复杂、不适合网格搜索的场景非常试用。SageMaker 超参优化可以在几百次尝试之内快速找到一组对应局部的高低点超参。

这部分实验将展示如何定义一个 SageMaker 超参优化任务，并快速找到交易策略（以双均线策略为例）历史上的最优参数组合。

In [ ]:
### 定义参数

我们首先定义环境变量：

In [9]:
import boto3

aws_account_id = boto3.client('sts').get_caller_identity().get('Account')
repository_name = 'sagemaker-hpo'
aws_region = 'us-east-1'
my_name = 'sailong-demo'
bucket = 'algo-trading-workshop-{}'.format(my_name)

ecr = boto3.client('ecr', region_name=aws_region)
s3 = boto3.client('s3', region_name=aws_region)

### 获取数据

定义一个函数，通过 Athena 获取行情数据：

In [10]:
!pip install awswrangler

import awswrangler as wr

s3_output = wr.athena.create_athena_bucket()

def execute_query(database, sql):
    
    query_execution_id = wr.athena.start_query_execution(database=database, sql=sql)
    response = wr.athena.get_query_execution(query_execution_id=query_execution_id)
    wr.athena.wait_query(query_execution_id=query_execution_id)
    OutputLocation = response['ResultConfiguration']['OutputLocation']
    
    return OutputLocation

You should consider upgrading via the '/home/ec2-user/anaconda3/envs/python3/bin/python -m pip install --upgrade pip' command.


In [11]:
database = 'stock-data'
table = 'stock_day'
fields = '*'
ticker = '600519'
orderby = 'tradedate'
sort = 'ASC'

sql = f'''
SELECT {fields}
FROM "{database}"."{table}"
WHERE ticker='{ticker}'
AND tradedate>='2019-01-01'
ORDER BY {orderby}
{sort}
'''

output_location = execute_query(database, sql)
print(output_location)

s3://aws-athena-query-results-661060674358-us-east-1/e17cef9a-cd46-4ea4-ac9c-683c31f239d6.csv


In [12]:
df = wr.s3.read_csv(path=[output_location])

df['ticker'] = df['ticker'].apply(lambda x: str(x))
df['ticker'] = df['ticker'].apply(lambda x: '0'*(6-len(x)) + x)
df['openprice'] = df['openprice'] * df['accumadjfactor'] / df['accumadjfactor'].iloc[-1]
df['closeprice'] = df['closeprice'] * df['accumadjfactor'] / df['accumadjfactor'].iloc[-1]
df['highestprice'] = df['highestprice'] * df['accumadjfactor'] / df['accumadjfactor'].iloc[-1]
df['lowestprice'] = df['lowestprice'] * df['accumadjfactor'] / df['accumadjfactor'].iloc[-1]
df = df[df['isopen'] == True]
df.drop('isopen', 1, inplace=True)
df.drop('accumadjfactor', 1, inplace=True)
df.set_index('tradedate', inplace=True)
df.sort_index(0, inplace=True)

df.rename(columns={'openprice': 'open'}, inplace=True)
df.rename(columns={'closeprice': 'close'}, inplace=True)
df.rename(columns={'highestprice': 'high'}, inplace=True)
df.rename(columns={'lowestprice': 'low'}, inplace=True)
df.rename(columns={'turnovervol': 'volume'}, inplace=True)
df['openinterest'] = 0 # A股回测中一般并不考虑利率，通常可以直接设为 0
df.head()

,ticker,secid,open,high,low,close,volume,turnovervalue,openinterest
tradedate,,,,,,,,,
2019-01-02,600519,600519.SH,588.598322,590.547514,574.153066,577.983905,6228649.0,3.754388e+09,0
2019-01-03,600519,600519.SH,578.939203,580.569963,565.265905,569.318682,3097735.0,1.838179e+09,0
2019-01-04,600519,600519.SH,566.694027,586.369294,561.618406,580.898045,3768347.0,2.258501e+09,0
2019-01-07,600519,600519.SH,586.687727,590.547514,581.110333,584.265710,3475013.0,2.105981e+09,0
2019-01-08,600519,600519.SH,584.275359,590.547514,579.238336,583.590247,2883813.0,1.750413e+09,0


将结果保存在这个实验专用的路径下，这个路径还将用于保存批量任务的输入和输出结果：

In [13]:
wr.s3.to_csv(df=df, path='s3://{}/{}/data.csv'.format(bucket, repository_name))

{'paths': ['s3://algo-trading-workshop-sailong-demo/sagemaker-hpo/data.csv'],
 'partitions_values': {}}

### 定义回测任务

使用 SageMaker 运行任务时，默认使用 /opt/ml/ 路径。任务运行时会默认运行 /opt/ml/code/ 路径下的 train 文件。因此我们需要对回测代码做一些简单改写：

In [16]:
! mkdir {repository_name}
! mkdir {repository_name}/model

In [18]:
%%writefile {repository_name}/model/train
#!/usr/bin/env python
from __future__ import (absolute_import, division, print_function,
                        unicode_literals)
from datetime import datetime
import json
import numpy as np
import pandas as pd
import os.path
import sys

import backtrader as bt

# 这里定义 SageMaker 任务默认的环境路径
prefix = '/opt/ml/'

config_path = os.path.join(prefix, 'input/config')
input_path = os.path.join(prefix, 'input/data/training')
data_path = os.path.join(input_path, 'data.csv')
output_path = os.path.join(prefix, 'output')
model_path = os.path.join(prefix, 'model')


class MyStrategy(bt.Strategy):
    ## 全局参数
    params=(('fastmaperiod', 10),
            ('slowmaperiod', 30),
            ('printlog', False),)

    ## 策略初始化
    def __init__(self):

        # 初始化交易指令、买卖价格和手续费
        self.order = None
        self.buyprice = None
        self.buycomm = None

        # 添加移动均线指标。Backtrader 集成了 talib，可以自动算出一些常见的技术指标
        self.fastma = bt.indicators.SimpleMovingAverage(self.datas[0], period=self.params.fastmaperiod)
        self.slowma = bt.indicators.SimpleMovingAverage(self.datas[0], period=self.params.slowmaperiod)
        
    ## 策略核心逻辑
    def next(self):
        # 记录收盘价
#         self.log('收盘价：%.2f' % self.datas[0].close[0])
        if self.order: # 检查是否有指令等待执行
            return
        # 检查是否持仓   
        if not self.position: # 如果没有持仓
            # 快线上穿慢线，执行买入
            if self.fastma[0] > self.slowma[0]:
                self.log('买入委托：%.2f' % self.datas[0].close[0])
                #执行买入
                self.size = int(self.broker.cash / self.datas[0].close[0])
                self.order = self.buy(size=self.size)
        else: # 
            # 快线下穿慢线，执行卖出
            if self.fastma[0] < self.slowma[0]:
                self.log('卖出委托：%.2f' % self.datas[0].close[0])
                #执行卖出
                self.order = self.sell(size=self.size)

    ## 4、日志记录
    # 交易记录日志（可选，默认不输出结果）
    def log(self, txt, dt=None, doprint=False):
        if self.params.printlog or doprint:
            dt = dt or self.datas[0].datetime.date(0)
            print(f'{dt.isoformat()},{txt}')

    # 记录交易执行情况（可选，默认不输出结果）
    def notify_order(self, order):
        # 如果 order 为 submitted/accepted，返回空
        if order.status in [order.Submitted, order.Accepted]:
            return
        # 如果 order 为 buy/sell executed，报告价格结果
        if order.status in [order.Completed]: 
            if order.isbuy():
                self.log(f'买入：\n价格：%.2f,\
                交易金额：-%.2f,\
                手续费：%.2f' % (order.executed.price, order.executed.value, order.executed.comm))
                self.buyprice = order.executed.price
                self.buycomm = order.executed.comm
            else:
                self.log(f'卖出:\n价格：%.2f,\
                交易金额：%.2f,\
                手续费：%.2f' % (order.executed.price, order.executed.price*self.size, order.executed.comm))
            self.bar_executed = len(self) 

        # 如果指令取消/交易失败, 报告结果
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('交易失败')
        self.order = None

    # 记录交易收益情况（可省略，默认不输出结果）
    def notify_trade(self,trade):
        if not trade.isclosed:
            return
        self.log(f'策略收益：\n毛收益 {trade.pnl:.2f}, 净收益 {trade.pnlcomm:.2f}')

    # 回测结束后输出结果（可省略，默认输出结果）
    def stop(self):
        self.log('(MA均线： %2d日  %2d日) 期末总资金 %.2f' %
                 (self.params.fastmaperiod, self.params.slowmaperiod, self.broker.getvalue()), doprint=True)

        
if __name__ == '__main__':
    
    # 创建 Cerebro 对象
    cerebro = bt.Cerebro()

    df = pd.read_csv(data_path)
    df.set_index('tradedate', inplace=True)
    
    # 创建 Data Feed
    df.index = pd.to_datetime(df.index)
    start = df.index[0]
    end = df.index[-1]
    print(start, '-', end)
    data = bt.feeds.PandasData(dataname=df, fromdate=start, todate=end)
    # 将 Data Feed 添加至 Cerebro
    cerebro.adddata(data)

    # 添加策略 Cerebro
    try:
        with open("{}/hyperparameters.json".format(config_path)) as json_file:
            params = json.load(json_file)
        print('Parameter load success')
    except Exception as e:
        print(e)
    cerebro.addstrategy(MyStrategy, fastmaperiod=int(params['fastmaperiod']), slowmaperiod=int(params['slowmaperiod']), printlog=True)
    
    # 设置初始资金
    cerebro.broker.setcash(100000.0)
    # 设置手续费为万二
    cerebro.broker.setcommission(commission=0.0002) 

    # 在开始时 print 初始账户价值
    print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

    # 运行回测流程
    cerebro.run()

    # 在结束时 print 最终账户价值
    print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())
    print('Return: %.4f' % (float(cerebro.broker.getvalue())/1e5 - 1))

    sys.exit(0)

Writing sagemaker-hpo/model/train


### 创建镜像仓库和容器镜像

接下来构建一个 python3 容器镜像和镜像仓库：

In [20]:
ecr.create_repository(repositoryName=repository_name)

RepositoryAlreadyExistsException: An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'sagemaker-hpo' already exists in the registry with id '661060674358'

In [21]:
%%writefile {repository_name}/Dockerfile
FROM python:3.8

RUN pip --no-cache-dir install \
    backtrader\
    boto3 \
    pandas

ENV PYTHONUNBUFFERED=TRUE
ENV PYTHONDONTWRITEBYTECODE=TRUE
ENV PATH="/opt/ml/code:${PATH}"

RUN mkdir -p /opt/ml/input/data
RUN mkdir -p /opt/ml/input/config
RUN mkdir -p /opt/ml/code
RUN mkdir -p /opt/ml/failure
RUN mkdir -p /opt/ml/model
RUN mkdir -p /opt/ml/output

COPY model /opt/ml/code
RUN chmod -R 777 /opt/ml/code
WORKDIR /opt/ml/code

Writing sagemaker-hpo/Dockerfile


In [22]:
!docker build {repository_name} -t {repository_name}

Sending build context to Docker daemon  9.216kB
Step 1/14 : FROM python:3.8
3.8: Pulling from library/python

765e08d1: Pulling fs layer 
670e5e55: Pulling fs layer 
180f4992: Pulling fs layer 
209566d0: Pulling fs layer 
44ed766e: Pulling fs layer 
520e0e66: Pulling fs layer 
83039b08: Pulling fs layer 
94c24f54: Pulling fs layer 
Digest: sha256:caa7d8d6bfaa181f30c5a5074b81b6963246615f0140dca1d86e1e98efa99dc6[9A
Status: Downloaded newer image for python:3.8
 ---> 1d136c1ca7e1
Step 2/14 : RUN pip --no-cache-dir install     backtrader    boto3     pandas
 ---> Running in 9c7e9909dd48
Removing intermediate container 9c7e9909dd48
 ---> e7b39bea9016
Step 3/14 : ENV PYTHONUNBUFFERED=TRUE
 ---> Running in 1a5d8882d00e
Removing intermediate container 1a5d8882d00e
 ---> 11ad55b890eb
Step 4/14 : ENV PYTHONDONTWRITEBYTECODE=TRUE
 ---> Running in 5630071f4870
Removing intermediate container 5630071f4870
 ---> 547f52870bae
Step 5/14 : ENV PATH="/opt/ml/code:${PATH}"
 ---> Running in f311a7a9bb88
R

SageMaker 提供了本地测试功能。在提交 HPO 任务之前，可以先在本地尝试运行镜像：

In [23]:
!pip uninstall -y sagemaker
!pip install sagemaker==1.72.0
import sagemaker
from sagemaker.estimator import Estimator

role = sagemaker.get_execution_role()
session = sagemaker.Session()

Found existing installation: sagemaker 2.45.0
Uninstalling sagemaker-2.45.0:
  Successfully uninstalled sagemaker-2.45.0
     |████████████████████████████████| 297 kB 32.3 MB/s eta 0:00:01
  Created wheel for sagemaker: filename=sagemaker-1.72.0-py2.py3-none-any.whl size=386358 sha256=fb60f316075ae7793011793cb516b1d0e2bf2cf93ef66978f56baa92abd4d0cf
  Stored in directory: /home/ec2-user/.cache/pip/wheels/c3/58/70/85faf4437568bfaa4c419937569ba1fe54d44c5db42406bbd7
Successfully built sagemaker
  Attempting uninstall: smdebug-rulesconfig
    Found existing installation: smdebug-rulesconfig 1.0.1
    Uninstalling smdebug-rulesconfig-1.0.1:
      Successfully uninstalled smdebug-rulesconfig-1.0.1
You should consider upgrading via the '/home/ec2-user/anaconda3/envs/python3/bin/python -m pip install --upgrade pip' command.


Exception ignored in: <bound method _S3ObjectBase.__del__ of <awswrangler.s3._fs._S3ObjectBase object at 0x7fc66ec23400>>
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/awswrangler/s3/_fs.py", line 305, in __del__
    self.close()
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/awswrangler/s3/_fs.py", line 545, in close
    function_name="put_object", s3_additional_kwargs=self._s3_additional_kwargs
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/awswrangler/_utils.py", line 331, in try_it
    return f(**kwargs)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/botocore/client.py", line 386, in _api_call
    return self._make_api_call(operation_name, kwargs)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/botocore/client.py", line 692, in _make_api_call
    operation_model, request_dict, request_context)
  File "/home/ec2-use

In [24]:
# 默认参数
params = { 
    'fastmaperiod': 6,
    'slowmaperiod': 11
}

estimator = Estimator(
    repository_name,
    role=role,
    train_instance_count=1,
    train_instance_type='local',
    output_path='s3://{}/{}/output'.format(bucket, repository_name),
    base_job_name=repository_name, 
    hyperparameters=params,
)

estimator.fit('s3://{}/{}/data.csv'.format(bucket, repository_name))

Parameter image_name will be renamed to image_uri in SageMaker Python SDK v2.
's3_input' class will be renamed to 'TrainingInput' in SageMaker Python SDK v2.


Creating tmppdbkowxv_algo-1-vepgd_1 ... 
Creating tmppdbkowxv_algo-1-vepgd_1 ... done
Attaching to tmppdbkowxv_algo-1-vepgd_1
algo-1-vepgd_1  | 2019-01-02 00:00:00 - 2021-07-05 00:00:00
algo-1-vepgd_1  | Parameter load success
algo-1-vepgd_1  | Starting Portfolio Value: 100000.00
algo-1-vepgd_1  | 2019-01-16,买入委托：637.93
algo-1-vepgd_1  | 2019-01-17,买入：
algo-1-vepgd_1  | 价格：639.76,                交易金额：-99802.53,                手续费：19.96
algo-1-vepgd_1  | 2019-02-26,卖出委托：701.85
algo-1-vepgd_1  | 2019-02-27,卖出:
algo-1-vepgd_1  | 价格：707.31,                交易金额：110339.75,                手续费：22.07
algo-1-vepgd_1  | 2019-02-27,策略收益：
algo-1-vepgd_1  | 毛收益 10537.22, 净收益 10495.19
algo-1-vepgd_1  | 2019-02-28,买入委托：728.54
algo-1-vepgd_1  | 2019-03-01,交易失败
algo-1-vepgd_1  | 2019-03-01,买入委托：761.63
algo-1-vepgd_1  | 2019-03-04,交易失败
algo-1-vepgd_1  | 2019-03-04,买入委托：754.45
algo-1-vepgd_1  | 2019-03-05,交易失败
algo-1-vepgd_1  | 2019-03-05,买入委托：752.45
algo-1-vepgd_1  | 2019-03-06,买入：
algo-1-vepgd_1  | 价格：7

测试成功之后再将镜像推送到 ECR：

In [25]:
!docker tag {repository_name} {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com/{repository_name}
!docker images
!aws ecr get-login-password | docker login --username AWS --password-stdin {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com
!docker push {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com/{repository_name}

REPOSITORY                                                   TAG                 IMAGE ID            CREATED             SIZE
661060674358.dkr.ecr.us-east-1.amazonaws.com/sagemaker-hpo   latest              11ae0ba7d82c        4 minutes ago       1.07GB
sagemaker-hpo                                                latest              11ae0ba7d82c        4 minutes ago       1.07GB
python                                                       3.8                 1d136c1ca7e1        2 days ago          884MB
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
The push refers to repository [661060674358.dkr.ecr.us-east-1.amazonaws.com/sagemaker-hpo]

e7ac35ed: Preparing 
f29e280e: Preparing 
4537d80a: Preparing 
b75c0b33: Preparing 
02bf922a: Preparing 
8fb65f84: Preparing 
6c8cd771: Preparing 
4df28046: 

### SageMaker 超参优化

接下来我们将定义一个超参优化任务。这首先，我们需要定义超参的类型和取值范围。在超参优化的过程中我们还需要定义一个目标参数（收益率）。SageMaker 将通过正则表达式从日志中提取目标参数所需的值，并判断最优的优化路径。这些配置信息通过 JSON 格式定义：

In [26]:
# 超参优化任务名称
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner

# 超参调优的参数配置
hyperparameter_ranges = {
    'fastmaperiod': IntegerParameter(5, 10),
    'slowmaperiod': IntegerParameter(10, 30),
}

# 目标参数的配置
metric_definitions = [
    {
        'Name': 'portfolio value', 
        'Regex': 'Final Portfolio Value: ([0-9\\.]+)'
    },
    {
        'Name': 'return', 
        'Regex': 'Return: ([0-9\\.]+)'
    }
]
objective_metric_name = "return"

In [27]:
estimator = Estimator(
    "{}.dkr.ecr.{}.amazonaws.com/{}".format(aws_account_id, aws_region, repository_name),
    role,
    train_instance_count=1,
    train_instance_type='ml.m5.2xlarge',
    output_path='s3://{}/{}/output'.format(bucket, repository_name),
    sagemaker_session=session
)

from sagemaker.tuner import HyperparameterTuner

tuner = HyperparameterTuner(
    estimator, 
    objective_metric_name,
    hyperparameter_ranges,
    metric_definitions=metric_definitions,
    strategy='Bayesian',
    objective_type='Maximize', 
    max_jobs=100,
    max_parallel_jobs=5,
    base_tuning_job_name=repository_name,
    early_stopping_type='Auto'
)

tuner.fit('s3://{}/{}/data.csv'.format(bucket, repository_name))

Parameter image_name will be renamed to image_uri in SageMaker Python SDK v2.
's3_input' class will be renamed to 'TrainingInput' in SageMaker Python SDK v2.


在超参优化任务结束之后，我们可以通过以下将结果加载到一个 DataFrame 中，并随时查看优化的结果：

In [28]:
tuner_analytics = tuner.analytics()
tuning_result = tuner_analytics.dataframe().sort_values(['FinalObjectiveValue'], ascending=False)
tuning_result.head(10)

,fastmaperiod,slowmaperiod,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
5,7.0,22.0,sagemaker-hpo-210809-0013-005-78e31cba,Completed,1.0040,2021-08-09 00:15:51+00:00,2021-08-09 00:16:46+00:00,55.0
8,9.0,19.0,sagemaker-hpo-210809-0013-002-83373bd1,Completed,0.9634,2021-08-09 00:15:41+00:00,2021-08-09 00:16:43+00:00,62.0
6,6.0,30.0,sagemaker-hpo-210809-0013-004-9e9ac4a1,Completed,0.8997,2021-08-09 00:15:59+00:00,2021-08-09 00:16:57+00:00,58.0
7,5.0,15.0,sagemaker-hpo-210809-0013-003-0b0f26eb,Completed,0.7914,2021-08-09 00:15:42+00:00,2021-08-09 00:16:33+00:00,51.0
9,10.0,27.0,sagemaker-hpo-210809-0013-001-9dbd0abe,Completed,0.6990,2021-08-09 00:15:38+00:00,2021-08-09 00:16:31+00:00,53.0
0,6.0,21.0,sagemaker-hpo-210809-0013-010-4a1dcb48,InProgress,NaN,NaT,NaT,NaN
1,5.0,22.0,sagemaker-hpo-210809-0013-009-cce53090,InProgress,NaN,NaT,NaT,NaN
2,6.0,22.0,sagemaker-hpo-210809-0013-008-5516b261,InProgress,NaN,NaT,NaT,NaN
3,9.0,20.0,sagemaker-hpo-210809-0013-007-02797bb4,InProgress,NaN,NaT,NaT,NaN
4,7.0,20.0,sagemaker-hpo-210809-0013-006-74268cdc,InProgress,NaN,NaT,NaT,NaN


可以观察下 SageMaker 是否很快就找到了最优参数组合呢？您可以将均线范围再扩大些，看看 SageMaker HPO 在实验组数更多的时候是否效果会比网格搜索更好